In [5]:
import pandas as pd
import numpy as np
import yaml
from linearmodels.panel import PanelOLS
import statsmodels.formula.api as smf



with open('../../Settings.yaml', 'r') as file:
    Setting = yaml.safe_load(file)

#Calling Dataset
file_name = "Adjusted.xlsx"
sheet_name = 'Dataset_for_Model'
file_path = f"{Setting['Output_Path_Ajusted']}/{file_name}"
df = pd.read_excel(file_path,sheet_name = sheet_name)

In [ ]:

# 1) (اختیاری) انتخاب کد صنعت به عنوان FE
industry_fe = "Industry_Code"   # یا "Industry_Name"

# 2) ساخت لگ‌ها
# اگر Blackout ممکن است صفر باشد، از log1p استفاده می‌کنیم: ln(1+x)
df = df.copy()

df["ln_sale"]        = np.log(df["Sale"])
df["ln_blackout"]    = np.log1p(df["Blackout"])        # ln(1+Blackout)
df["ln_wage"]        = np.log(df["Wage"])
df["ln_input_value"] = np.log(df["Input_Value"])
df["ln_electricity"] = np.log(df["Electricity"])

# 3) حذف ردیف‌های مشکل‌دار (صفر/منفی برای متغیرهایی که log گرفتیم + NaN/inf)
needed = ["ln_sale","ln_blackout","ln_wage","ln_input_value","ln_electricity","Year",industry_fe]
df_reg = (df[needed]
          .replace([np.inf, -np.inf], np.nan)
          .dropna()
         )

# 4) رگرسیون با اثر ثابت سال و صنعت
# ln(Sale_it) = a + b ln(1+Blackout_it) + gX_it + Year FE + Industry FE + e_it
formula = f"ln_sale ~ ln_blackout + ln_wage + ln_input_value + ln_electricity + C(Year) + C({industry_fe})"

# (پیشنهاد) خطای استاندارد خوشه‌ای روی صنعت (یا اگر شناسه بنگاه داری، روی بنگاه)
res = smf.ols(formula, data=df_reg).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg[industry_fe]}
)

print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                ln_sale   R-squared:                       0.973
Model:                            OLS   Adj. R-squared:                  0.971
Method:                 Least Squares   F-statistic:                     1129.
Date:                Fri, 24 Apr 2026   Prob (F-statistic):           6.12e-30
Time:                        18:24:40   Log-Likelihood:                -34.226
No. Observations:                 504   AIC:                             160.5
Df Residuals:                     458   BIC:                             354.7
Df Model:                          45                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 15

c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 46, but rank is 22
  warnings.warn('covariance of constraints does not have full '


In [9]:
import numpy as np
import statsmodels.formula.api as smf

industry_fe = "Industry_Code"   # یا "Industry_Name"

df = df.copy()

# لگ‌ها
df["ln_sale"]        = np.log(df["Sale"])
df["ln_blackout"]    = np.log1p(df["Blackout"])   # ln(1+Blackout) برای حالت وجود صفر
df["ln_wage"]        = np.log(df["Wage"])
df["ln_electricity"] = np.log(df["Electricity"])

# پاکسازی
needed = ["ln_sale","ln_blackout","ln_wage","ln_electricity", industry_fe]
df_reg = (df[needed]
          .replace([np.inf, -np.inf], np.nan)
          .dropna()
         )

# فقط اثر ثابت صنعت (اثر زمان حذف شد)
formula = f"ln_sale ~ ln_blackout + ln_wage + ln_electricity + C({industry_fe})"

res = smf.ols(formula, data=df_reg).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg[industry_fe]}
)

print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                ln_sale   R-squared:                       0.960
Model:                            OLS   Adj. R-squared:                  0.958
Method:                 Least Squares   F-statistic:                     11.70
Date:                Fri, 24 Apr 2026   Prob (F-statistic):           7.43e-05
Time:                        18:23:44   Log-Likelihood:                -132.65
No. Observations:                 504   AIC:                             319.3
Df Residuals:                     477   BIC:                             433.3
Df Model:                          26                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 18

c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 26, but rank is 3
  warnings.warn('covariance of constraints does not have full '
